In [1]:
import pandas as pd
import re
from typing import Dict

In [2]:
"""
Script para extraer entidades biomédicas de abstracts
Extrae: SPECIES, TAXON, GENE, PROTEIN, CHEMICAL, DRUG, DISEASE, CONDITION
"""


def extraer_entidades_biomedicas(texto: str) -> Dict:
    """
    Extracción de entidades biomédicas usando patrones mejorados
    Para producción avanzada, considerar usar BioBERT, SciBERT o PubMedBERT
    """
    
    if not texto or pd.isna(texto):
        return {
            'SPECIES': [],
            'TAXON': [],
            'GENE': [],
            'PROTEIN': [],
            'CHEMICAL': [],
            'DRUG': [],
            'DISEASE': [],
            'CONDITION': []
        }
    
    texto_lower = texto.lower()
    entidades = {
        'SPECIES': [],
        'TAXON': [],
        'GENE': [],
        'PROTEIN': [],
        'CHEMICAL': [],
        'DRUG': [],
        'DISEASE': [],
        'CONDITION': []
    }
    
    # SPECIES y TAXON (organismos)
    especies_patron = r'\b(human|humans|mouse|mice|rat|rats|E\.\s*coli|Escherichia\s+coli|' \
                     r'SARS-CoV-2|HIV|Pseudomonas\s+aeruginosa|Staphylococcus\s+aureus|' \
                     r'Candida\s+albicans|Salmonella|Mycobacterium|bacterial|bacteria|' \
                     r'viral|virus|fungal|yeast|mammalian|primate|rodent)\b'
    especies = list(set(re.findall(especies_patron, texto, re.IGNORECASE)))
    entidades['SPECIES'] = especies
    entidades['TAXON'] = especies  # En este contexto simplificado, son similares
    
    # GENES (patrones comunes en nomenclatura genética)
    # Genes suelen estar en mayúsculas con números: TP53, BRCA1, etc.
    gene_patron = r'\b([A-Z]{2,}[0-9]+[A-Z]*|[A-Z][a-z]{2,}[0-9]+)\b'
    posibles_genes = re.findall(gene_patron, texto)
    # Filtrar palabras comunes que no son genes
    palabras_excluir = {'DNA', 'RNA', 'PCR', 'ATP', 'USA', 'UK', 'PMC', 'PMID'}
    genes_filtrados = [g for g in posibles_genes if g not in palabras_excluir and len(g) <= 10]
    entidades['GENE'] = list(set(genes_filtrados))[:15]  # Limitar a 15 más relevantes
    
    # PROTEINS (proteínas comunes)
    proteinas_patron = r'\b(protein|enzyme|antibody|kinase|receptor|ligand|cytokine|' \
                      r'immunoglobulin|albumin|collagen|hemoglobin|insulin|interferon|' \
                      r'interleukin|tumor necrosis factor|TNF|growth factor)\b'
    entidades['PROTEIN'] = list(set(re.findall(proteinas_patron, texto, re.IGNORECASE)))
    
    # CHEMICALS (compuestos químicos)
    quimicos_patron = r'\b(glucose|sodium|potassium|calcium|chloride|phosphate|sulfate|' \
                     r'hydrogen peroxide|oxygen|carbon dioxide|nitric oxide|ATP|ADP|' \
                     r'amino acid|nucleotide|lipid|carbohydrate|steroid|peptide|' \
                     r'compound|molecule|ion|acid|base)\b'
    entidades['CHEMICAL'] = list(set(re.findall(quimicos_patron, texto, re.IGNORECASE)))
    
    # DRUGS (medicamentos y tratamientos)
    drogas_patron = r'\b(aspirin|ibuprofen|acetaminophen|penicillin|amoxicillin|' \
                   r'ciprofloxacin|metformin|insulin|warfarin|heparin|morphine|' \
                   r'antibiotic|antimicrobial|antiviral|antifungal|chemotherapy|' \
                   r'immunosuppressant|vaccine|drug|medication|therapy|treatment|' \
                   r'therapeutic agent|pharmaceutical)\b'
    entidades['DRUG'] = list(set(re.findall(drogas_patron, texto, re.IGNORECASE)))
    
    # DISEASE (enfermedades específicas)
    enfermedades_patron = r'\b(cancer|carcinoma|tumor|leukemia|lymphoma|melanoma|' \
                         r'diabetes|hypertension|asthma|pneumonia|tuberculosis|malaria|' \
                         r'COVID-19|influenza|HIV/AIDS|hepatitis|sepsis|stroke|' \
                         r'Alzheimer|Parkinson|arthritis|osteoporosis|anemia)\b'
    entidades['DISEASE'] = list(set(re.findall(enfermedades_patron, texto, re.IGNORECASE)))
    
    # CONDITIONS (condiciones médicas generales)
    condiciones_patron = r'\b(infection|inflammation|disease|disorder|syndrome|' \
                        r'deficiency|dysfunction|failure|injury|trauma|lesion|' \
                        r'chronic|acute|pathology|mortality|morbidity|complication)\b'
    entidades['CONDITION'] = list(set(re.findall(condiciones_patron, texto, re.IGNORECASE)))
    
    return entidades


def procesar_csv_abstracts(archivo_entrada: str, columna_abstract: str = 'Abstract') -> pd.DataFrame:
    """
    Procesa un CSV con abstracts y extrae entidades biomédicas
    """
    # Leer CSV
    print(f"Leyendo archivo: {archivo_entrada}")
    df = pd.read_csv(archivo_entrada)
    
    if columna_abstract not in df.columns:
        raise ValueError(f"Columna '{columna_abstract}' no encontrada en el CSV")
    
    print(f"Total de registros: {len(df)}")
    print(f"Columnas disponibles: {', '.join(df.columns)}")
    print(f"\nProcesando abstracts...\n")
    
    resultados = []
    
    for idx, row in df.iterrows():
        print(f"Procesando {idx + 1}/{len(df)}...", end='\r')
        
        abstract = row[columna_abstract] if pd.notna(row[columna_abstract]) else ""
        
        # Extraer entidades
        entidades = extraer_entidades_biomedicas(abstract)
        
        # Crear registro con todas las columnas originales
        resultado = row.to_dict()
        
        # Agregar entidades como columnas nuevas
        resultado.update({
            'species': ', '.join(entidades['SPECIES']),
            'taxon': ', '.join(entidades['TAXON']),
            'genes': ', '.join(entidades['GENE']),
            'proteins': ', '.join(entidades['PROTEIN']),
            'chemicals': ', '.join(entidades['CHEMICAL']),
            'drugs': ', '.join(entidades['DRUG']),
            'diseases': ', '.join(entidades['DISEASE']),
            'conditions': ', '.join(entidades['CONDITION'])
        })
        
        resultados.append(resultado)
    
    print(f"\nProcesamiento completado!")
    
    # Crear DataFrame con resultados
    df_resultados = pd.DataFrame(resultados)
    
    return df_resultados


def mostrar_estadisticas(df: pd.DataFrame):
    """
    Muestra estadísticas de las entidades extraídas
    """
    print("\n" + "="*60)
    print("ESTADÍSTICAS DE EXTRACCIÓN")
    print("="*60)
    
    entidades_cols = ['species', 'taxon', 'genes', 'proteins', 'chemicals', 'drugs', 'diseases', 'conditions']
    
    for col in entidades_cols:
        if col in df.columns:
            con_entidades = df[col].apply(lambda x: len(str(x).strip()) > 0).sum()
            porcentaje = (con_entidades / len(df)) * 100
            print(f"{col.upper():15} : {con_entidades:4}/{len(df)} ({porcentaje:.1f}%)")
    
    print("="*60)


if __name__ == "__main__":
    # CONFIGURACIÓN
    archivo_entrada = "articles_data_updated.csv"
    archivo_salida = "pubmed_data3.csv"
    columna_abstract = "Abstract"  # Nombre de la columna con los abstracts
    
    try:
        # Procesar
        df_resultados = procesar_csv_abstracts(archivo_entrada, columna_abstract)
        
        # Guardar resultados
        df_resultados.to_csv(archivo_salida, index=False)
        
        # Mostrar estadísticas
        mostrar_estadisticas(df_resultados)
        
        print(f"\n✓ Resultados guardados en: {archivo_salida}")
        print(f"✓ Total de artículos procesados: {len(df_resultados)}")
        print(f"✓ Columnas en el archivo de salida: {len(df_resultados.columns)}")
        
        # Mostrar ejemplo de las primeras 3 filas
        print("\nEjemplo de primeras entidades extraídas:")
        print("-" * 60)
        for idx in range(min(3, len(df_resultados))):
            print(f"\nArtículo {idx + 1}:")
            if 'diseases' in df_resultados.columns and df_resultados.iloc[idx]['diseases']:
                print(f"  Diseases: {df_resultados.iloc[idx]['diseases'][:100]}...")
            if 'genes' in df_resultados.columns and df_resultados.iloc[idx]['genes']:
                print(f"  Genes: {df_resultados.iloc[idx]['genes'][:100]}...")
        
    except FileNotFoundError:
        print(f"\n❌ Error: No se encontró el archivo '{archivo_entrada}'")
        print("Asegúrate de que el archivo existe en el directorio actual.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

Leyendo archivo: articles_data_updated.csv
Total de registros: 607
Columnas disponibles: PMC_ID, Title, Authors, Introduction, Development/Methods, Results, Discussion, References_Count, References, Conclusions, Abstract

Procesando abstracts...

Procesando 607/607...
Procesamiento completado!

ESTADÍSTICAS DE EXTRACCIÓN
SPECIES         :  367/607 (60.5%)
TAXON           :  367/607 (60.5%)
GENES           :  143/607 (23.6%)
PROTEINS        :  174/607 (28.7%)
CHEMICALS       :  150/607 (24.7%)
DRUGS           :  104/607 (17.1%)
DISEASES        :   40/607 (6.6%)
CONDITIONS      :  165/607 (27.2%)

✓ Resultados guardados en: pubmed_data3.csv
✓ Total de artículos procesados: 607
✓ Columnas en el archivo de salida: 19

Ejemplo de primeras entidades extraídas:
------------------------------------------------------------

Artículo 1:

Artículo 2:

Artículo 3:


In [3]:
df = pd.read_csv("pubmed_data3.csv")
df.head()

,PMC_ID,Title,Authors,Introduction,Development/Methods,Results,Discussion,References_Count,References,Conclusions,Abstract,species,taxon,genes,proteins,chemicals,drugs,diseases,conditions
0,PMC4136787,Mice in Bion-M 1 Space Mission: Training and S...,Alexander Andreev-Andrievskiy; Anfisa Popova; ...,"After a 16-year hiatus, Russia resumed in 2013...",The study was approved by IACUC of MSU Institu...,Living conditions for animals considered optim...,Living conditions for animals considered optim...,37,2007 Animals in space Vestnik Rossijskoj Akade...,The microbiome of the Gowanus Canal is a biote...,We investigate the bioremediation potential of...,NaN,NaN,NaN,NaN,NaN,antimicrobial,NaN,NaN
1,PMC3630201,Microgravity Induces Pelvic Bone Loss through ...,Elizabeth A. Blaber; Natalya Dvorochkin; Chial...,"On Earth, at 1 g, mechanical loading of mammal...",All experimental animal procedures for STS-131...,All flight and ground control mice were observ...,"In this study, we investigated cellular and mo...",74,2000 Historical overview of the Bion project J...,As the volume of spaceflight omics-level data ...,Recent advances in the routine access to space...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PMC11988870,Microgravity and Cellular Biology: Insights in...,Nelson Adolfo López Garzón; María Virginia Pin...,"Microgravity, a condition characterized by min...",A comprehensive literature review was conducte...,NaN,Recent research demonstrates that microgravity...,70,2003 Genetic models in applied physiology: sel...,NaN,The oocytes of the African clawed frog (Xenopu...,bacterial,bacterial,NaN,protein,ion,NaN,NaN,NaN
3,PMC7998608,Selective Proliferation of Highly Functional A...,Takanobu Mashiko; Koji Kanayama; Natsumi Saito...,Human adipose-derived stem cells (hASCs) are e...,Human lipoaspirates were obtained from 12 heal...,Cells were expanded for three passages before ...,"Through novel advances in cell biology, adult ...",48,2013 Effects of spaceflight and ground recover...,Plasmids Size (bp) Inc group GC% N° ORFs Start...,"Extra-intestinal pathogenicE. coli(ExPEC), inc...",human,human,NaN,NaN,acid,NaN,NaN,infection
4,PMC5587110,Microgravity validation of a novel system for ...,Macarena Parra; Jimmy Jung; Travis D. Boone; L...,The ISS National Laboratory is a unique resear...,"In order to validate the system, a number of g...",In order to assess the functionality of PCR in...,One of the major obstacles to space exploratio...,38,2013 Changes in Mouse Thymus and Spleen after ...,Spaceflight poses risks to the central nervous...,Spaceflight poses risks to the central nervous...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Save the updated DataFrame
df.to_csv("articles_data_updated.csv", index=False, encoding='utf-8')

In [7]:
dfr = pd.read_csv("articles_data_updated.csv")
dfr.head(10)

,PMC_ID,Title,Authors,Introduction,Development/Methods,Results,Discussion,References_Count,References,Conclusions,Abstract,species,taxon,genes,proteins,chemicals,drugs,diseases,conditions
0,PMC4136787,Mice in Bion-M 1 Space Mission: Training and S...,Alexander Andreev-Andrievskiy; Anfisa Popova; ...,"After a 16-year hiatus, Russia resumed in 2013...",The study was approved by IACUC of MSU Institu...,Living conditions for animals considered optim...,Living conditions for animals considered optim...,37,2007 Animals in space Vestnik Rossijskoj Akade...,The microbiome of the Gowanus Canal is a biote...,We investigate the bioremediation potential of...,NaN,NaN,NaN,NaN,NaN,antimicrobial,NaN,NaN
1,PMC3630201,Microgravity Induces Pelvic Bone Loss through ...,Elizabeth A. Blaber; Natalya Dvorochkin; Chial...,"On Earth, at 1 g, mechanical loading of mammal...",All experimental animal procedures for STS-131...,All flight and ground control mice were observ...,"In this study, we investigated cellular and mo...",74,2000 Historical overview of the Bion project J...,As the volume of spaceflight omics-level data ...,Recent advances in the routine access to space...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PMC11988870,Microgravity and Cellular Biology: Insights in...,Nelson Adolfo López Garzón; María Virginia Pin...,"Microgravity, a condition characterized by min...",A comprehensive literature review was conducte...,NaN,Recent research demonstrates that microgravity...,70,2003 Genetic models in applied physiology: sel...,NaN,The oocytes of the African clawed frog (Xenopu...,bacterial,bacterial,NaN,protein,ion,NaN,NaN,NaN
3,PMC7998608,Selective Proliferation of Highly Functional A...,Takanobu Mashiko; Koji Kanayama; Natsumi Saito...,Human adipose-derived stem cells (hASCs) are e...,Human lipoaspirates were obtained from 12 heal...,Cells were expanded for three passages before ...,"Through novel advances in cell biology, adult ...",48,2013 Effects of spaceflight and ground recover...,Plasmids Size (bp) Inc group GC% N° ORFs Start...,"Extra-intestinal pathogenicE. coli(ExPEC), inc...",human,human,NaN,NaN,acid,NaN,NaN,infection
4,PMC5587110,Microgravity validation of a novel system for ...,Macarena Parra; Jimmy Jung; Travis D. Boone; L...,The ISS National Laboratory is a unique resear...,"In order to validate the system, a number of g...",In order to assess the functionality of PCR in...,One of the major obstacles to space exploratio...,38,2013 Changes in Mouse Thymus and Spleen after ...,Spaceflight poses risks to the central nervous...,Spaceflight poses risks to the central nervous...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PMC8396460,Spaceflight Modulates the Expression of Key Ox...,Akhilesh Kumar; Candice G. T. Tahimic; Eduardo...,Responses to spaceflight include cardiovascula...,All animal procedures were conducted in accord...,"Immediately after landing, all flight (FLT) an...",Our findings provide new insight into how the ...,76,2009 Effects of spaceflight on innate immune f...,NaN,Efforts to understand the impact of spacefligh...,"mice, mouse, human","mice, mouse, human",PI3K,NaN,NaN,NaN,NaN,NaN
6,PMC5666799,Dose- and Ion-Dependent Effects in the Oxidati...,Joshua S. Alwood; Luan H. Tran; Ann-Sofie Schr...,Structural degradation and oxidative stress fo...,"Male C57BL/6J mice (Jackson Laboratories, Bar ...",To evaluate the individual effects of radiatio...,Heavy-ion irradiation during space missions is...,57,2006 Reduced susceptibility to ventricular tac...,NaN,Spaceflight has been shown to suppress the ada...,"mice, Rodent","mice, Rodent",NaN,Immunoglobulin,NaN,treatment,NaN,NaN
7,PMC5460236,From the bench to exploration medicine: NASA l...,Joshua S. Alwood; April E. Ronca; Richard C. M...,With the International Space Station (ISS) ava...,NaN,NaN,NaN,87,2012 The Mice Drawer System (MDS) Experiment a...,NaN,Excessive weight gain in adults is associated ...,"rat, virus, rats, Rats","rat, virus, rats, Rats",IGF1,protein,NaN,therapy,NaN,NaN
8,PMC6222041,Hi